# Tarea 2: GANs, VAEs y NFs
## MDS7203 Modelos Generativos Profundos
### Magíster en Ciencia de Datos - Universidad de Chile
#### Primavera, 2025

---

**Estudiante:** _[Nombre Completo]_

---

## Instrucciones generales

Esta tarea está compuesta por dos partes. En la primera parte se pide implementar un VAE sobre el dataset CelebA para luego hacer modificación de atributos. En la segunda parte, se busca crear melodías usando una GAN (más precisamente, una versión de MuseGAN). En esta tarea no se evaluarán contenidos asociados a flujos normalizantes (NF) ya que estos serán revisitados al momento de estudiar modelos generativos a tiempo continuo en la tarea 3.

- El trabajo es individual y debe ser realizado en Google Colab.
- Se pueden usar libremente herramientas como ChatGPT, Gemini o Claude.
- Se debe entregar este notebook completamente ejecutado, y debe poder volver a ejecutarse sin ningún problema.

**Las partes tienen distinta ponderación**. Los puntajes son los siguientes:
- Parte 1 (VAE): 2 puntos.
- Parte 2 (GAN): 4 puntos.

In [ ]:
!apt-get install -y timidity

import os
import json
import subprocess

import numpy as np
import pandas as pd
from PIL import Image
from matplotlib import pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import music21
from IPython.display import Audio, display

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando {DEVICE}.')

## Parte 1: Modificación de atributos usando un VAE

El objetivo de esta primera parte es modificar algunos atributos de imágenes de caras, utilizando un VAE entrenado sobre el dataset CelebA. Dado que este problema fue estudiado e implementado en clases, solo se pedirá implementar:
- Una red neuronal convolucional que incluya un mecanismo de atención en ciertas capas.
- El procedimiento para realizar la modificación de atributos.

Las implementaciones para cargar el dataset CelebA y entrenar el VAE vienen dadas, al igual que los hiperparámetros de entrenamiento.

### Parte 1.1: Datos de entrenamiento

Se comenzará descargando el dataset desde Kaggle. Para esto, es necesario obtener una [key de acceso a Kaggle](https://www.kaggle.com/settings) de forma similar a lo realizado en la tarea 1 para obtener acceso a los modelos LLaMA en Hugging Face:

In [ ]:
USERNAME = ''
KEY = ''

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)

with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': USERNAME, 'key': KEY}, f)

os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!kaggle datasets download -d jessicali9530/celeba-dataset --unzip -p celeba

Con los datos descargados, se implementará el dataset manualmente:

In [ ]:
class CelebA(Dataset):

    def __init__(self, img_dir='celeba/img_align_celeba/img_align_celeba/', attr_file='celeba/list_attr_celeba.csv', max_samples=None, img_size=64, seed=0):
        self.img_dir = img_dir
        self.img_size = img_size

        # Cargar atributos desde CSV:
        self.attributes_df = pd.read_csv(attr_file)
        self.attribute_names = list(self.attributes_df.columns[1:])

        # Filtrar imágenes que tienen atributos disponibles:
        available_images = set(self.attributes_df['image_id'].values)
        all_images = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        self.image_files = [f for f in all_images if f in available_images]

        # Submuestreo aleatorio si se especifica max_samples:
        if max_samples is not None and max_samples < len(self.image_files):
            rng = np.random.default_rng(seed)
            indices = rng.choice(len(self.image_files), size=max_samples, replace=False)
            self.image_files = [self.image_files[i] for i in sorted(indices)]

        # Transformaciones para preprocesar imágenes:
        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.CenterCrop(img_size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.img_dir, img_name)

        # Cargar y transformar imagen:
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)

        # Obtener atributos de la imagen (convertir -1/1 a 0/1):
        row = self.attributes_df[self.attributes_df['image_id'] == img_name].iloc[0]
        attributes = [(row[attr] + 1) // 2 for attr in self.attribute_names]

        return image, attributes

In [ ]:
dataset = CelebA(max_samples=50000, img_size=64)
dataloader = DataLoader(dataset, batch_size=128, shuffle=True, drop_last=True)

La siguiente función permite visualizar una imagen con sus atributos:

In [ ]:
def show_sample(image, attributes, attribute_names):
    plt.figure(figsize=(2, 2))
    plt.imshow(image.permute(1, 2, 0))
    plt.axis('off')
    plt.show()

    present = [name for name, val in zip(attribute_names, attributes) if val == 1]
    print(f'\nAtributos presentes ({len(present)}):', ', '.join(present), '\n')

In [ ]:
# Ejemplo 1:
img, attrs = dataset[0]
show_sample(img, attrs, dataset.attribute_names)

# Ejemplo 2:
img, attrs = dataset[1]
show_sample(img, attrs, dataset.attribute_names)

### Parte 1.2: Red neuronal convolucional con mecanismo de atención

> __[PREGUNTA 1 (1 punto)]__ Implemente la clase `AttentionVAE` asociada a la red neuronal de un VAE incondicional. La red neuronal debe incluir mecanismos de atención de forma similar a lo que se realiza en self-attention GAN (SAGAN). Es decir, en los bloques convolucionales donde se decida incluir un mecanismo de atención, la atención se calcula tratando cada pixel como un token distinto, donde la dimensión de embedding corresponde a la cantidad de canales que tiene el bloque convolucional donde se está aplicando el mecanismo de atención. Si desea, puede definir antes un módulo `SelfAttention` con el mecanismo de atención convolucional.

Recordar que el método `forward` debe retornar el par $(\mu_\theta,\sigma_\theta)\in\mathbb{R}^L\times\mathbb{R}^L_{++}$ asociado al encoder y el tensor $r_\phi\in[0,1]^D$ asociado al decoder (revisar [implementación vista en clases](https://github.com/fernando-fetis/MDS7203/blob/main/2025%2C%20primavera/Clases/Clase%2014/notebooks/VAE.ipynb)). La red neuronal debe ser una red convolucional acorde a la complejidad del dataset, y puede usar cualquiera de las técnicas vistas durante el curso.



In [ ]:
class AttentionVAE(nn.Module):

    def __init__(self, latent_dim=128):
        super().__init__()
        ...

    def forward(self, x):
        ...

        return (mu, std), x_dec

In [ ]:
# Ejemplo de uso:

vae = AttentionVAE(latent_dim=128).to(DEVICE)
batch = next(iter(dataloader))
images = batch[0].to(DEVICE)
(mu, std), x_dec = vae(images)

assert mu.shape == (images.shape[0], 128)
assert std.shape == (images.shape[0], 128)
assert x_dec.shape == images.shape

### Parte 1.3: Entrenamiento y reconstrucción

Teniendo definida la red neuronal a utilizar, se entrenará siguiendo el enfoque clásico de maximizar la ELBO. La siguiente función `train_vae` (idéntica a la vista en clases) se encarga de esto:

In [ ]:
def train_vae(model, dataloader, n_epochs=10):

    optimizer = torch.optim.Adam(model.parameters())
    model = model.to(DEVICE)
    model.train()

    try:
        for epoch in range(1, n_epochs + 1):
            for x, _ in tqdm(dataloader, desc=f'Época {epoch}/{n_epochs}'):
                x = x.to(DEVICE)
                (mu, std), x_dec = model(x)
                reconstruction = F.binary_cross_entropy(x_dec, x, reduction='sum')
                prior_matching = - 0.5 * torch.sum(1 + 2 * std.log() - mu.pow(2) - std.pow(2))
                loss = reconstruction + prior_matching
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

    except KeyboardInterrupt:
        print('\nEntrenamiento interrumpido.')

    finally:
        torch.save(model.state_dict(), 'vae_celeba.pth')
        print('Parámetros del VAE guardados en vae_celeba.pth.')

In [ ]:
model = AttentionVAE()
train_vae(model, dataloader)

Para revisar el entrenamiento, se probará la tarea de reconstrucción usando el VAE entrenado.

Primero se cargarán los parámetros del entrenamiento:

In [ ]:
model = AttentionVAE()
model.load_state_dict(torch.load('vae_celeba.pth', map_location=DEVICE))

Luego, se puede realizar la tarea de reconstrucción de manera usual:

In [ ]:
def reconstruction(x, model):
    model.to(DEVICE)
    model.eval()
    x = x.to(DEVICE)
    with torch.no_grad():
        (_, _), x_dec = model(x)
    return x_dec.cpu()

imgs, _ = next(iter(dataloader))

x_original = imgs[:6]
x_reconstruction = reconstruction(x_original.to(DEVICE), model)

# Visualización:
fig, axes = plt.subplots(2, 6, figsize=(12, 4))
for i in range(6):
    axes[0, i].imshow(x_original[i].permute(1, 2, 0))
    axes[0, i].axis('off')
    axes[0, i].set_title('Original', fontsize=10)

    axes[1, i].imshow(x_reconstruction[i].permute(1, 2, 0))
    axes[1, i].axis('off')
    axes[1, i].set_title('Reconstrucción', fontsize=10)

plt.tight_layout()
plt.show()

### Parte 1.4: Aritmética en el espacio latente

El objetivo de esta última parte es realizar aritmética de atributos, donde los atributos, en este caso, vienen dados en la lista `attributes` de cada instancia del dataset.

Recordar que, para un atributo dado, se puede calcular un centroide de dicho atributo, $z_{\text{atributo}}\in\mathbb{R}^L$, promediando representaciones latentes de muestras que posean dicho atributo. Con esto, restando los centroides de dos atributos diferentes se obtiene un vector latente director, $z_{\text{director}}\in\mathbb{R}^L$, el cual puede usarse para modificar atributos de una muestra $x\in\mathbb{R}^D$ mediante $x_\lambda = \text{decoder}(\text{encoder}(x)+\lambda z_\text{director})$, donde $\lambda>0$ es un parámetro de traslación.

> __[PREGUNTA 2 (1 punto)]__ Implemente las funciones `compute_centroid` y `attribute_arithmetic`.

In [ ]:
def compute_centroid(model, dataset, attribute_name, n_samples=1000):
    '''
    Calcula el centroide en el espacio latente para un atributo dado.

    Args:
        model: VAE entrenado
        dataset: Dataset de CelebA
        attribute_name: Nombre del atributo (ej: 'Male', 'Smiling')
        n_samples: Número de muestras a usar para calcular el centroide

    Returns:
        Tensor con el centroide (promedio de representaciones latentes)
    '''
    ...


def attribute_arithmetic(model, dataset, original_image, attr_1, attr_2, ponderators):
    '''
    Realiza aritmética de atributos en el espacio latente.

    Args:
        model: VAE entrenado
        dataset: Dataset de CelebA
        original_image: Imagen original (tensor)
        attr_1: Atributo fuente (string)
        attr_2: Atributo objetivo (string)
        ponderators: Lista de valores lambda para la interpolación

    Returns:
        Lista de imágenes modificadas
    '''
    ...

Para probar la implementación, se realizarán dos ejemplos. El primer ejemplo consistirá en agregar sonrisa a la imagen de una persona que no esté sonriendo:

In [ ]:
# Buscar una imagen sin sonrisa:
for i in range(len(dataset)):
    img, attrs = dataset[i]
    smiling_idx = dataset.attribute_names.index('Smiling')
    if attrs[smiling_idx] == 0:
        original_img = img
        break

ponderators = [-0.5, 0, 0.5, 1.0, 1.5, 2.0]
modified_images = attribute_arithmetic(model, dataset, original_img, attr_1='No_Beard', attr_2='Smiling', ponderators=ponderators)

# Visualización:
fig, axes = plt.subplots(1, len(ponderators) + 1, figsize=(14, 2))
axes[0].imshow(original_img.permute(1, 2, 0).numpy())
axes[0].set_title('Original', fontsize=10)
axes[0].axis('off')

for i, (img, lam) in enumerate(zip(modified_images, ponderators)):
    axes[i + 1].imshow(img.permute(1, 2, 0).numpy())
    axes[i + 1].set_title(f'λ={lam}', fontsize=10)
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

Como segundo ejemplo se modificará una imagen masculina para que tenga características femeninas:

In [ ]:
# Buscar una imagen masculina:
for i in range(len(dataset)):
    img, attrs = dataset[i]
    male_idx = dataset.attribute_names.index('Male')
    if attrs[male_idx] == 1:
        original_img_male = img
        break

ponderators = [-0.5, 0, 0.5, 1.0, 1.5, 2.0]
modified_images_gender = attribute_arithmetic(model, dataset, original_img_male, attr_1='Male', attr_2='Heavy_Makeup', ponderators=ponderators)

# Visualización:
fig, axes = plt.subplots(1, len(ponderators) + 1, figsize=(14, 2))
axes[0].imshow(original_img_male.permute(1, 2, 0).numpy())
axes[0].set_title('Original (Male)', fontsize=10)
axes[0].axis('off')

for i, (img, lam) in enumerate(zip(modified_images_gender, ponderators)):
    axes[i + 1].imshow(img.permute(1, 2, 0).numpy())
    axes[i + 1].set_title(f'λ={lam}', fontsize=10)
    axes[i + 1].axis('off')

plt.tight_layout()
plt.show()

## Parte 2: Generación de melodías usando una GAN

En esta segunda parte de la tarea se busca implementar una versión simplificada del modelo [MuseGAN (Dong et al., 2017)](https://arxiv.org/abs/1709.06298). Este modelo consiste en generar melodías MIDI tratando a las melodías como imágenes, donde los canales de color ahora toman el papel de _voces_ (o instrumentos distintos). Antes de ver los detalles de esta arquitectura, es necesario entender bien el funcionamiento de los archivos MIDI y la forma en la que se trabajarán las melodías.

### Parte 2.1: Dataset de entrenamiento

El modelo MuseGAN permite aprender distribuciones asociadas a melodías polifónicas de varios canales de voz y de varios compases musicales.

El dataset que se utilizará está compuesto por 382 preludios corales compuestos por J.S. Bach. Cada una de estas melodías puede variar entre 84 tonos distintos (pitches), a través de 4 voces (tracks): soprano, alto, tenor y bajo. Para efectos de reproducción, estas voces serán representadas con melodías generadas por un violín, una viola, un violonchelo y un contrabajo respectivamente.

Dado que el modelo que se implementará debe fijar la cantidad de compases a generar, se considerarán únicamente los primeros 2 compases de cada melodía para disminuir el costo de entrenamiento. Sin embargo, este hiperparámetro se puede ajustar en la definición de la clase `MusicDataset`. Además, se asumirá que cada compás del dataset tendrá 16 timesteps (más precisamente, cada compás tiene 4 tiempos, donde la unidad mínima de tiempo es la semicorchea). Esto significa que, para cada compás, se podrán definir 16 _instantes melódicos_ distintos.

Por lo tanto, cada batch de entrenamiento tendrá un tamaño `[batch, n_bars, n_steps_per_bar, n_pitches, n_tracks]`, donde `batch` es el tamaño del batch, `n_bars=2` es la cantidad de compases de cada melodía, `n_steps_per_bar=16` es la cantidad de steps que tendrá cada compás, `n_pitches=84` representará (mediante one-hot encoding) los pitches que estarán sonando, y `n_tracks=4` es la cantidad de voces (tracks) de cada melodía.

La siguiente clase organiza todo esto:

In [ ]:
class MusicDataset(Dataset):
    '''Dataset para chorales de Bach en formato piano roll'''

    def __init__(self, data_path, n_bars=2, n_steps_per_bar=16, max_pitch=83):
        self.n_bars = n_bars
        self.n_steps_per_bar = n_steps_per_bar
        self.max_pitch = max_pitch
        self.n_pitches = max_pitch + 1

        # Cargar melodías:
        chorales = np.load(data_path, allow_pickle=True)  # lista de arrays de tamaño variable (timesteps, n_tracks).
        n_samples = len(chorales)
        self.n_tracks = chorales[0].shape[1]

        # Considerar solo los primeros n_bars compases:
        total_steps = n_steps_per_bar * n_bars
        chorales_trimmed = np.array([chorale[:total_steps] for chorale in chorales])  # [n_samples, total_steps, n_tracks]
        chorales_clean = np.nan_to_num(chorales_trimmed, nan=max_pitch).astype(int)

        # Reorganizar en compases:
        chorales_reshaped = chorales_clean.reshape(n_samples, n_bars, n_steps_per_bar, self.n_tracks)

        # One-hot encoding para los pitches (tonos):
        chorales_onehot = np.eye(self.n_pitches)[chorales_reshaped]  # [n_samples, n_bars, n_steps_per_bar, n_tracks, n_pitches]

        # Normalización a [-1, 1] (el generador utiliza tanh en la salida):
        chorales_onehot[chorales_onehot == 0] = -1

        # Reordenar dimensiones:
        chorales_reordered = chorales_onehot.transpose(0, 1, 2, 4, 3)  # [n_samples, n_bars, n_steps_per_bar, n_pitches, n_tracks]

        self.data = torch.FloatTensor(chorales_reordered)

        print(f'Shape del dataset: {self.data.shape}')
        print(f'{n_samples} muestras, {n_bars} compases, {n_steps_per_bar} pasos por compás, {self.n_pitches} pitches, {self.n_tracks} tracks.')

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

    def convert_to_midi(self, piano_roll_data, output_path):
        '''Convierte una muestra del dataset (o la salida del generador) a un archivo MIDI y lo reproduce.'''

        # piano_roll_data.shape: [n_bars, n_steps_per_bar, n_pitches, n_tracks].

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Extraer pitches (argmax del one-hot encoding):
        pitches = torch.argmax(piano_roll_data, dim=-2)  # [n_bars, n_steps_per_bar, n_tracks]

        # Aplanar compases:
        total_steps = self.n_bars * self.n_steps_per_bar
        pitch_sequence = pitches.reshape(total_steps, self.n_tracks)  # [total_steps, n_tracks]

        # Crear partitura:
        score = music21.stream.Score()
        score.append(music21.tempo.MetronomeMark(number=66))

        # Instrumentos de cuerda para las 4 voces (SATB):
        instruments = [
            music21.instrument.Violin(),       # soprano (voz más aguda).
            music21.instrument.Viola(),        # alto (voz intermedia).
            music21.instrument.Violoncello(),  # tenor (voz grave).
            music21.instrument.Contrabass()    # bajo (voz más grave).
        ]

        for track_idx in range(self.n_tracks):
            part = music21.stream.Part()
            part.insert(0, instruments[track_idx])

            current_pitch = int(pitch_sequence[0, track_idx].item())
            current_duration = 0.0

            for step_idx in range(len(pitch_sequence)):
                pitch = int(pitch_sequence[step_idx, track_idx].item())
                if (pitch != current_pitch or step_idx % 4 == 0) and step_idx > 0:
                    part.append(music21.note.Note(current_pitch, quarterLength=current_duration))
                    current_pitch = pitch
                    current_duration = 0.0
                current_duration += 0.25

            part.append(music21.note.Note(current_pitch, quarterLength=current_duration))
            score.append(part)

        # Guardar MIDI y convertir a WAV:
        score.write('midi', fp=output_path)
        wav_file = output_path.replace('.midi', '.wav')
        subprocess.run(['timidity', output_path, '-Ow', '-o', wav_file], check=True, capture_output=True)
        os.remove(output_path)

        # Reproducir audio:
        display(Audio(wav_file))

In [ ]:
# Inicializar dataset:
dataset = MusicDataset('music_data.npy')
dataloader = DataLoader(dataset, batch_size=len(dataset), shuffle=True, drop_last=True)

# Ejemplo de melodía:
sample = dataset.data[0]
dataset.convert_to_midi(sample, 'output/dataset_sample.midi')

> __[PREGUNTA 1 (1 punto)]__ Entienda completamente el funcionamiento de la clase `MusicDataset`. En particular, explique detalladamente el funcionamiento del método `convert_to_midi`.

### Parte 2.2: Redes neuronales

Las GANs son modelos de variable latente, lo que permite realizar modificación de atributos de alto nivel manipulando la representación latente de las muestras. Como se vio al estudiar los VAEs, una propiedad deseable del espacio latente es que tenga factores desacoplados (p.g. un $\beta$-VAE lo logra dándole más peso al término de prior matching $\operatorname{{D}_{KL}}\left(q_\phi(z|x)\| p_\theta(z)\right)$) ya que esto permite modificar atributos específicos de la generación mientras el resto de atributos se mantienen invariantes.

Sin embargo, no es obvio cómo hacer esto en una GAN ya que la función de entrenamiento (maximizar/minimizar la verosimilitud del discriminador $q_\phi(y|x)$) no permite incluir directamente un sesgo inductivo para esta propiedad. Para aminorar esta limitación en la generación de audio, la arquitectura MuseGAN propone construir la variable latente $z\in\mathbb{R}^{4L}$ que recibe el generador concatenando 4 vectores latentes independientes:

- $z_1\in\mathbb{R}^L$ (style latent): es un vector latente común para todas las tracks y para todos los compases de la melodía.
- $z_2\in\mathbb{R}^L$ (chord latent): es un vector latente común para todas las tracks, pero es pasado por una red neuronal para tomar representaciones distintas en cada compás.
- $z_3\in\mathbb{R}^L$ (groove latent): es un vector latente distinto para cada una de las tracks, pero común para todos los compases de la melodía.
- $z_4\in\mathbb{R}^L$ (melody latent): es un vector latente distinto para cada una de las tracks, y es pasado por una red neuronal para tomar representaciones distintas en cada compás.

De este modo, la variable latente que recibe el generador de la GAN está compuesto por fragmentos que forman las 4 combinaciones posibles (común/independiente entre los compases, y común/independiente entre las tracks). El siguiente diagrama muestra este funcionamiento para generar el 2º compás de la 3º track (recordar que nuestro dataset tiene 4 voces y 2 compases por muestra):

<img src="MuseGAN_diagram.png" width="700"/>

Se observa que para obtener vectores latentes distintos en cada track (caso de groove latent y melody latent), basta con generar 4 vectores latentes distintos (uno para cada track). En cambio, para obtener vectores latentes distintos en cada compás (caso de chord latent y melody latent), el vector latente asociado al track que se busca generar es pasado por una red neuronal (llamada `TemporalNetwork`) que expande dicho vector según la cantidad de compases que se busca generar.

En cambio, los vectores comunes se comparten entre todas las tracks y compases. En particular, se utiliza el mismo chord latent en la generación de todas las tracks (para un compás fijo), mientras que se utiliza el mismo groove latent en la generación de todos los compases (para un track fijo). El vector style latent es compartido en la generación de todas las tracks y de todas los compases.

__Importante:__ los nombres dados a cada componente latente (style, chord, groove y melody) no tienen por qué representar los conceptos asociados a los nombres (de hecho, estos son términos vagos que no tienen una definición precisa) ya que en ninguna parte se está forzando a la red neuronal a aprender dichas características. Sin embargo, estos nombres los eligieron los autores de MuseGAN ya que se asocian a los factores latentes desacoplados que identificaron una vez la red neuronal estaba entrenada.

La red neuronal `TemporalNetwork` usada para expandir un vector latente según la cantidad de compases que se busca generar (en nuestro caso, 2 compases para cada track) es una red convolucional transpuesta ya que este tipo de redes neuronales permite aumentar la _resolución_ de la entrada:

In [ ]:
class TemporalNetwork(nn.Module):
    '''Red temporal para expandir un vector latente en múltiples compases.'''

    def __init__(self, z_dim, n_bars):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(z_dim, 1024),  # [batch, z_dim] -> [batch, 1024]
            nn.BatchNorm1d(1024),
            nn.ReLU()
        )
        self.conv = nn.Sequential(
            nn.ConvTranspose2d(1024, 1024, kernel_size=(2, 1), stride=(1, 1), padding=0),  # [batch, 1024, 1, 1] -> [batch, 1024, 2, 1]
            nn.BatchNorm2d(1024),
            nn.ReLU(),
            nn.ConvTranspose2d(1024, z_dim, kernel_size=(n_bars - 1, 1), stride=(1, 1), padding=0),  # [batch, 1024, 2, 1] -> [batch, z_dim, n_bars, 1]
            nn.BatchNorm2d(z_dim),
            nn.ReLU()
        )

    def forward(self, z):
        x = self.fc(z)  # [batch, z_dim] -> [batch, 1024]
        x = x.view(x.size(0), 1024, 1, 1)  # [batch, 1024, 1, 1]
        x = self.conv(x)  # [batch, z_dim, n_bars, 1]
        x = x.squeeze(-1).transpose(1, 2)  # [batch, n_bars, z_dim]
        return x

In [ ]:
# Ejemplo de uso:

temporal_net = TemporalNetwork(z_dim=128, n_bars=2).to(DEVICE)
z_sample = torch.randn(4, 128).to(DEVICE)
output = temporal_net(z_sample)

assert output.shape == (4, 2, 128)

De forma similar, la red neuronal `BarGenerator` usada para generar compases a partir del vector latente $z=\text{Concat}(z_1,z_2,z_3,z_4)\in\mathbb{R}^{4L}$ (asociado al compás y track que se busca generar) también es una red neuronal convolucional transpuesta, lo cual es lo usual para la generación de imágenes (en este caso, compases melódicos) a partir de una representación latente:

In [ ]:
class BarGenerator(nn.Module):
    '''Generador de un compas individual a partir de un vector latente asociado a un track y compás específico.'''

    def __init__(self, z_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(z_dim * 4, 1024),  # [batch, z_dim * 4] -> [batch, 1024]
            nn.BatchNorm1d(1024),
            nn.ReLU()
        )

        self.conv = nn.Sequential(
            nn.ConvTranspose2d(512, 512, kernel_size=(2, 1), stride=(2, 1), padding=(0, 0)),  # [batch, 512, 2, 1] -> [batch, 512, 4, 1]
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.ConvTranspose2d(512, 256, kernel_size=(2, 1), stride=(2, 1), padding=(0, 0)),  # [batch, 512, 4, 1] -> [batch, 256, 8, 1]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 256, kernel_size=(2, 1), stride=(2, 1), padding=(0, 0)),  # [batch, 256, 8, 1] -> [batch, 256, 16, 1]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 256, kernel_size=(1, 7), stride=(1, 7), padding=(0, 0)),  # [batch, 256, 16, 1] -> [batch, 256, 16, 7]
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 1, kernel_size=(1, 12), stride=(1, 12), padding=(0, 0)),  # [batch, 256, 16, 7] -> [batch, 1, 16, 84]
            nn.Tanh()
        )

    def forward(self, z):
        x = self.fc(z)  # [batch, z_dim * 4] -> [batch, 1024]
        x = x.view(x.size(0), 512, 2, 1)  # [batch, 512, 2, 1]
        x = self.conv(x)  # [batch, 1, n_steps_per_bar, n_pitches]
        return x

In [ ]:
# Ejemplo de uso:

bar_net = BarGenerator(z_dim=128).to(DEVICE)
z_bar_sample = torch.randn(4, 128 * 4).to(DEVICE)
bar_output = bar_net(z_bar_sample)

assert bar_output.shape == (4, 1, 16, 84)

Antes de implementar la red neuronal generadora completa (la que genera todos los compases de todas las voces) se definirá la siguiente función `weights_init`, la cual permite inicializar los parámetros de una red neuronal a partir de una distribución gaussiana de baja varianza (recomendación de DCGAN):

In [ ]:
def weights_init(m):
    '''Inicializar pesos con distribucion normal.'''
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d, nn.Conv3d, nn.Linear)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

Teniendo definidas las redes neuronales `TemporalNetwork` y `BarGenerator`, lo único que queda pendiente para la generación es juntar las piezas para poder generar de forma ordenada todos los compases de todas las tracks (voces). La forma de hacer esto es la siguiente:

1. Se expande chord latent usando una `TemporalNetwork`. Esto permite obtener una representación latente para cada compás, la cual se comparte en todas las tracks.
2. Se expande cada melody latent usando una `TemporalNetwork` (una red neuronal `TemporalNetwork` distinta por track). Esto permite obtener una representación latente para cada compás de cada track.
3. Iterar sobre la generación de compases para cada track. Es decir, para cada compás que se busca generar, y para cada track que se busca generar, se forma el vector latente $z=\text{Concat}(z_1,z_2,z_3,z_4)\in\mathbb{R}^{4L}$ correspondiente, el cual es pasado por `BarGenerator` (una red neuronal `BarGenerator` distinta por track).
4. Todos los compases de todas las tracks son guardados en un tensor de tamaño `[batch, n_bars, n_steps_per_bar, n_pitches, n_tracks]`, el cual representa la melodía completa generada. Este tensor es retornado por el método `forward`.

> __[PREGUNTA 2 (2 puntos)]__ Implemente el método `forward` de la red neuronal `Generator`.

In [ ]:
class Generator(nn.Module):
    '''Generador de MuseGAN'''

    def __init__(self, z_dim, n_bars, n_steps_per_bar, n_pitches, n_tracks):
        super().__init__()
        self.n_bars = n_bars
        self.n_tracks = n_tracks
        self.n_steps_per_bar = n_steps_per_bar
        self.n_pitches = n_pitches

        # Red temporal para acordes:
        self.chords_network = TemporalNetwork(z_dim, n_bars)

        # Redes temporales para melodías (una por cada track):
        self.melody_networks = nn.ModuleList([TemporalNetwork(z_dim, n_bars) for _ in range(n_tracks)])

        # Redes generadoras de compases (una por cada track):
        self.bar_generators = nn.ModuleList([BarGenerator(z_dim) for _ in range(n_tracks)])

        self.apply(weights_init)

    def forward(self, z_style, z_chord, z_groove, z_melody):
        '''
        Args:
            z_style: [batch, z_dim]
            z_chord: [batch, z_dim]
            z_groove: [batch, n_tracks, z_dim]
            z_melody: [batch, n_tracks, z_dim]
        '''

        ...

In [ ]:
# Ejemplo de uso:

generator = Generator(z_dim=128, n_bars=2, n_steps_per_bar=16, n_pitches=84, n_tracks=4).to(DEVICE)
style_z = torch.randn(4, 128).to(DEVICE)
chords_z = torch.randn(4, 128).to(DEVICE)
groove_z = torch.randn(4, 4, 128).to(DEVICE)
melody_z = torch.randn(4, 4, 128).to(DEVICE)
generated_music = generator(style_z, chords_z, groove_z, melody_z)

assert generated_music.shape == (4, 2, 16, 84, 4)

Teniendo el generador de la GAN entrenado, solo queda pendiente definir la red neuronal discriminadora. La siguiente red neuronal `Critic` cumple esta función. El nombre `Critic` (en vez de `Discriminator`) se debe a que se utilizará una formulación alternativa de la GAN, llamada _Wasserstein GAN_ (WGAN), donde el cambio principal estará en la función de pérdida a utilizar durante el entrenamiento. Además, en una WGAN, la red neuronal auxiliar (discriminador en una GAN clásica) ya no tiene estructura de clasificador, por lo que no es correcto llamarla "discriminador". En particular, se observa que no se está aplicando la función logística (sigmoide) en la salida de `Critic`, por lo que su salida no puede interpretarse como una probabilidad de clasificación.

In [ ]:
class Critic(nn.Module):
    '''Critic de MuseGAN para WGAN-GP'''

    def __init__(self, n_bars, n_steps_per_bar, n_pitches, n_tracks):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv3d(n_tracks, 128, kernel_size=(2, 1, 1), stride=(1, 1, 1), padding=(0, 0, 0)),  # [batch, n_tracks, n_bars, n_steps, n_pitches] -> [batch, 128, n_bars-1, n_steps, n_pitches]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 128, kernel_size=(n_bars-1, 1, 1), stride=(1, 1, 1), padding=(0, 0, 0)),  # [batch, 128, n_bars-1, n_steps, n_pitches] -> [batch, 128, 1, n_steps, n_pitches]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 128, kernel_size=(1, 1, 12), stride=(1, 1, 12), padding=(0, 0, 6)),  # [batch, 128, 1, n_steps, n_pitches] -> [batch, 128, 1, n_steps, n_pitches//12]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 128, kernel_size=(1, 1, 7), stride=(1, 1, 7), padding=(0, 0, 3)),  # [batch, 128, 1, n_steps, n_pitches//12] -> [batch, 128, 1, n_steps, n_pitches//84]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 128, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 1, 0)),  # [batch, 128, 1, n_steps, n_pitches//84] -> [batch, 128, 1, n_steps//2, n_pitches//84]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 128, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 1, 0)),  # [batch, 128, 1, n_steps//2, n_pitches//84] -> [batch, 128, 1, n_steps//4, n_pitches//84]
            nn.LeakyReLU(0.2),
            nn.Conv3d(128, 256, kernel_size=(1, 4, 1), stride=(1, 2, 1), padding=(0, 2, 0)),  # [batch, 128, 1, n_steps//4, n_pitches//84] -> [batch, 256, 1, n_steps//8, n_pitches//84]
            nn.LeakyReLU(0.2),
            nn.Conv3d(256, 512, kernel_size=(1, 3, 1), stride=(1, 2, 1), padding=(0, 1, 0)),  # [batch, 256, 1, n_steps//8, n_pitches//84] -> [batch, 512, 1, n_steps//16, n_pitches//84]
            nn.LeakyReLU(0.2)
        )

        # Calcular tamano despues de convoluciones:
        with torch.no_grad():
            dummy_input = torch.zeros(1, n_tracks, n_bars, n_steps_per_bar, n_pitches)
            dummy_output = self.conv(dummy_input)
            conv_output_size = dummy_output.view(1, -1).size(1)

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(conv_output_size, 1024),  # [batch, conv_output_size] -> [batch, 1024]
            nn.LeakyReLU(0.2),
            nn.Linear(1024, 1)  # [batch, 1024] -> [batch, 1]
        )

        self.apply(weights_init)

    def forward(self, x):
        x = x.permute(0, 4, 1, 2, 3)  # [batch, n_bars, n_steps_per_bar, n_pitches, n_tracks] -> [batch, n_tracks, n_bars, n_steps_per_bar, n_pitches]
        x = self.conv(x)
        x = self.fc(x)  # [batch, 1]
        return x

In [ ]:
# Ejemplo de uso:

critic = Critic(n_bars=2, n_steps_per_bar=16, n_pitches=84, n_tracks=4).to(DEVICE)
sample_music = torch.randn(4, 2, 16, 84, 4).to(DEVICE)
critic_output = critic(sample_music)

assert critic_output.shape == (4, 1)

### Parte 2.3: Entrenamiento y generación

Como se comentó anteriormente, los autores de MuseGAN entrenan las redes neuronales utilizando el enfoque de una WGAN en vez de una GAN clásica. Si bien esta formulación tiene implicancias profundas (p.g. conexión con transporte óptimo), la principal diferencia práctica entre ambos enfoques es la función de costo a utilizar. En particular, `Critic` (que cumple el papel del discriminador clásico) es entrenado mediante

$$
\max_D \; \mathbb{E}_{x\sim p_{\text{data}}(x)}[D(x)] \;-\; \mathbb{E}_{z\sim p_z(z)}[D(G(z))]
$$

mientras que `Generator` es entrenado mediante

$$
\min_G \; -\mathbb{E}_{z\sim p_z}[D(G(z))]
$$

Más aún, la formulación de WGAN requiere que el `Critic` $D:\mathbb{R}^D\to\mathbb{R}$ sea 1-Lipschitz, lo que en la práctica se puede implementar truncando los pesos de la red neuronal para que no excedan un cierto umbral predefinido. Sin embargo, esta forma de implementar la condición 1-Lipschitz es inestable, por lo que se prefiere utilizar penalización sobre los gradientes de $D$.

La siguiente clase `MusicGAN` implementa todo lo necesario para entrenar y generar muestras desde una WGAN:

In [ ]:
class MusicGAN:
    '''Wrapper para MuseGAN con funcionalidades de entrenamiento y generacion'''

    def __init__(self, dataset, z_dim=32):
        self.dataset = dataset
        self.z_dim = z_dim
        self.generator = Generator(z_dim, dataset.n_bars, dataset.n_steps_per_bar, dataset.n_pitches, dataset.n_tracks).to(DEVICE)
        self.critic = Critic(dataset.n_bars, dataset.n_steps_per_bar, dataset.n_pitches, dataset.n_tracks).to(DEVICE)


    def train(self, dataloader, epochs=6000, critic_steps=5, gp_weight=10, grad_clip=1.0):
        '''Entrena MuseGAN con WGAN-GP.'''

        g_optimizer = optim.Adam(self.generator.parameters(), lr=0.001, betas=(0.5, 0.9))
        c_optimizer = optim.Adam(self.critic.parameters(), lr=0.001, betas=(0.5, 0.9))

        self.generator.train()
        self.critic.train()

        try:
            progress_bar = tqdm(range(1, epochs + 1), desc='Training')

            for epoch in progress_bar:
                for batch_data in dataloader:

                    batch_data = batch_data.to(DEVICE)
                    batch_size = batch_data.size(0)

                    # ---------- entrenamiento del critic ----------
                    for _ in range(critic_steps):

                        # Generación de datos falsos:
                        z_style = torch.randn(batch_size, self.z_dim, device=DEVICE)
                        z_chords = torch.randn(batch_size, self.z_dim, device=DEVICE)
                        z_groove = torch.randn(batch_size, self.dataset.n_tracks, self.z_dim, device=DEVICE)
                        z_melody = torch.randn(batch_size, self.dataset.n_tracks, self.z_dim, device=DEVICE)
                        fake_data = self.generator(z_chords, z_style, z_melody, z_groove)

                        # Scores del critic:
                        real_score = self.critic(batch_data)
                        fake_score = self.critic(fake_data.detach())

                        # Función de pérdida del critic:
                        wasserstein_loss = fake_score.mean() - real_score.mean()
                        gp = self._gradient_penalty(batch_data, fake_data.detach())
                        critic_loss = wasserstein_loss + gp_weight * gp

                        # Optimización (con gradient clipping):
                        c_optimizer.zero_grad()
                        critic_loss.backward()
                        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), grad_clip)
                        c_optimizer.step()

                    # ---------- Entrenamiento del generador ----------

                    # Generación de datos falsos:
                    z_style = torch.randn(batch_size, self.z_dim, device=DEVICE)
                    z_chords = torch.randn(batch_size, self.z_dim, device=DEVICE)
                    z_groove = torch.randn(batch_size, self.dataset.n_tracks, self.z_dim, device=DEVICE)
                    z_melody = torch.randn(batch_size, self.dataset.n_tracks, self.z_dim, device=DEVICE)
                    fake_data = self.generator(z_chords, z_style, z_melody, z_groove)

                    # Función de pérdida del generador:
                    fake_score = self.critic(fake_data)
                    generator_loss = -fake_score.mean()

                    # Optimización (con gradient clipping):
                    g_optimizer.zero_grad()
                    generator_loss.backward()
                    torch.nn.utils.clip_grad_norm_(self.generator.parameters(), grad_clip)
                    g_optimizer.step()

                    progress_bar.set_postfix({'G_Loss': f'{generator_loss:.4f}', 'C_Loss': f'{critic_loss:.4f}'})

        except KeyboardInterrupt:
            print('Entrenamiento interrumpido.')

        finally:
            torch.save(self.generator.state_dict(), 'musegan_generator.pth')
            print('Parámetros del generador guardados en musegan_generator.pth.')


    def _gradient_penalty(self, real_data, fake_data):
        '''Calcula gradient penalty para WGAN-GP'''

        # real_data, fake_data: [batch, n_bars, n_steps_per_bar, n_pitches, n_tracks] cada uno.

        batch_size = real_data.size(0)

        # Interpolar entre datos reales y falsos:
        alpha = torch.rand(batch_size, 1, 1, 1, 1, device=DEVICE)
        interpolated = alpha * real_data + (1 - alpha) * fake_data  # [batch, n_bars, n_steps_per_bar, n_pitches, n_tracks]
        interpolated.requires_grad_(True)

        critic_output = self.critic(interpolated)  # [batch, 1]

        # Gradientes con respecto a las entradas interpoladas:
        gradients = torch.autograd.grad(  # [batch, n_bars, n_steps_per_bar, n_pitches, n_tracks]
            outputs=critic_output,
            inputs=interpolated,
            grad_outputs=torch.ones_like(critic_output),
            create_graph=True,
            retain_graph=True
        )[0]

        gradient_norm = torch.sqrt(torch.sum(gradients ** 2, dim=[1, 2, 3, 4]))  # [batch]
        penalty = ((gradient_norm - 1) ** 2).mean()  # escalar.

        return penalty


    @torch.no_grad()
    def generate(self, z_chords=None, z_style=None, z_melody=None, z_groove=None,
                 filename='generated', output_dir='output'):
        '''Genera musica con vectores latentes opcionales'''
        self.generator.eval()

        # Vectores latentes:
        z_style = z_style.to(DEVICE) if z_style is not None else torch.randn(1, self.z_dim, device=DEVICE)
        z_chords = z_chords.to(DEVICE) if z_chords is not None else torch.randn(1, self.z_dim, device=DEVICE)
        z_groove = z_groove.to(DEVICE) if z_groove is not None else torch.randn(1, self.dataset.n_tracks, self.z_dim, device=DEVICE)
        z_melody = z_melody.to(DEVICE) if z_melody is not None else torch.randn(1, self.dataset.n_tracks, self.z_dim, device=DEVICE)

        # Generación:
        generated = self.generator(z_chords, z_style, z_melody, z_groove)

        # Guardar como MIDI y WAV:
        midi_path = os.path.join(output_dir, f'{filename}')
        dataset.convert_to_midi(generated[0], midi_path)

        # Retornar vectores latentes usados (para modificación de atributos):
        return z_style, z_chords, z_groove, z_melody

> __[PREGUNTA 3 (1 punto)]__ Averigüe acerca de la formulación de una WGAN y la penalización por gradiente (implementada en el método `_gradient_penalty`) que realiza la variante WGAN-GP. En particular, explique la relación entre la función de pérdida implementada y la distancia de Wasserstein entre la distribución aprendida por la GAN y la distribución real de los datos.

Teniendo todas las redes neuronales y la clase `MusicGAN` definida, se entrenará el modelo sobre el dataset de melodías definido anteriormente:

In [ ]:
musegan = MusicGAN(dataset)

print('Número de parámetros:')
print(f'- Generator: {sum(p.numel() for p in musegan.generator.parameters()):,}')
print(f'- Critic: {sum(p.numel() for p in musegan.critic.parameters()):,}')

In [ ]:
musegan.train(dataloader, epochs=1000)

Una vez entrenado el modelo, se cargarán los parámetros del generador guardados al final del entrenamiento, y se generará una melodía de ejemplo:

In [ ]:
# Cargar generador entrenado:
musegan.generator.load_state_dict(torch.load('musegan_generator.pth', map_location=DEVICE))

# Generación:
z_style, z_chords, z_groove, z_melody = musegan.generate(filename='generated_music.midi')

Por otro lado, es posible generar variantes de la melodía recién creada reutilizando algunas de las variables latentes que se usaron para su generación:

In [ ]:
# Variación de z_style:
_ = musegan.generate(z_chords=z_chords, z_groove=z_groove, z_melody=z_melody, filename='variation_z_style.midi')

# Variación de z_chords:
_ = musegan.generate(z_style=z_style, z_groove=z_groove, z_melody=z_melody, filename='variation_z_chords.midi')

# Variación de z_groove:
_ = musegan.generate(z_style=z_style, z_chords=z_chords, z_melody=z_melody, filename='variation_z_groove.midi')

# Variación de z_melody:
_ = musegan.generate(z_style=z_style, z_chords=z_chords, z_groove=z_groove, filename='variation_z_melody.midi')